# Non-DOI Harvest

Harvesting datasets metadata for responsitories not issuing DOIs

## Import

In [1]:
%load_ext autoreload
%autoreload 2

In [9]:
import os
from sindex.sources.emdb.normalize import slim_emdb_record
from sindex.sources.emdb.jobs import batch_slim_emdb_record_to_ndjson

## Electron Microscopy Data Bank (EMDB)

### Get EMDB record using api

In [13]:
emdb_record_folder = os.path.join(base_path, "emdb-records")
emdb_record_path= os.path.join(emdb_record_folder, "emdb-records.ndjson")

In [12]:
n = batch_harvest_emdb("2025-09-30", emdb_record_path)
print("Total written:", n)

Fetching EMDB IDs from CSV search endpoint...


KeyboardInterrupt: 

### Create slim metadata record

In [4]:
emdb_record_path = r"D:\pipeline-data\records\raw-records\emdb-records\emdb-records.ndjson"

#### Test with one record

In [10]:
with open(emdb_record_path, 'r', encoding='utf-8') as f:
    first_line = f.readline()
    if first_line:
        first_record = json.loads(first_line)
        print("First record:", first_record)

First record: {'_id': '6743ad09c7dd9684974495f4', 'admin': {'authors_list': {'author': [{'instance_type': 'author', 'valueOf_': 'Markert J'}, {'instance_type': 'author', 'valueOf_': 'Farnung L'}]}, 'current_status': {'code': {'valueOf_': 'REL'}, 'date': '2025-11-26T00:00:00', 'processing_site': 'RCSB'}, 'grant_support': {'grant_reference': [{'country': 'United States', 'funding_body': 'National Institutes of Health/National Institute of Environmental Health Sciences (NIH/NIEHS)', 'instance_type': 'grant_reference'}, {'country': 'United States', 'funding_body': 'Richard and Susan Smith Family Foundation', 'instance_type': 'grant_reference'}, {'country': 'United States', 'funding_body': 'Damon Runyon Cancer Research Foundation', 'instance_type': 'grant_reference'}, {'country': 'United States', 'funding_body': 'Rita Allen Foundation', 'instance_type': 'grant_reference'}]}, 'key_dates': {'deposition': '2024-11-21T00:00:00', 'header_release': '2025-11-26T00:00:00', 'map_release': '2025-11-2

In [11]:
slim_record = slim_emdb_record(metadata = first_record)

In [12]:
display(slim_record)

{'source': 'emdb',
 'identifiers': [{'identifier': 'EMD-48024', 'identifier_type': 'emdb_id'}],
 'url': 'https://www.ebi.ac.uk/emdb/EMD-48024',
 'title': 'map beta Paf1C',
 'subjects': ['Transcription', 'SETD2', 'H3K36me3'],
 'publication_date': '2024-11-21T00:00:00',
 'publication_year': 2024,
 'creators': [{'name': 'Markert J', 'name_type': 'Personal'},
  {'name': 'Farnung L', 'name_type': 'Personal'}],
 'publisher': 'The Electron Microscopy Data Bank (EMDB)'}

### Batch slim record

In [18]:
raw_emdb_record_folder = r"D:\pipeline-data\records\raw-records\emdb-records"
slim_emdb_record_folder = r"D:\pipeline-data\records\slim-records\emdb-slim-records"

In [19]:
summary = batch_slim_emdb_record_to_ndjson(
    src_folder=raw_emdb_record_folder,
    dst_folder=slim_emdb_record_folder,
    overwrite=True,
    one_line_progress=True
)

[1/1] files completed
Done. files=1 kept=51,645 bad=0 time=13.6s rate≈3,807/s → D:\pipeline-data\records\slim-records\emdb-slim-records


### Check for the different instance types for authors

In [24]:
def scan_author_instance_types_with_examples(src_folder: str, accept_gz: bool = True):
    """
    Scan EMDB NDJSON files and collect all unique author instance types,
    along with one example valueOf_ (or raw name for string authors).

    Returns:
        dict mapping:
            instance_type -> example_name
    """
    from pathlib import Path
    import gzip
    import json

    results = {}  # instance_type -> example name

    src = Path(src_folder)

    patterns = ["*.ndjson"]
    if accept_gz:
        patterns.append("*.ndjson.gz")

    files = []
    for pat in patterns:
        files.extend(src.glob(pat))

    for f in files:
        opener = (
            (lambda p: gzip.open(p, "rt", encoding="utf-8"))
            if f.suffix == ".gz"
            else (lambda p: open(p, "rt", encoding="utf-8"))
        )

        with opener(f) as r:
            for line in r:
                try:
                    rec = json.loads(line)
                except Exception:
                    continue

                admin = rec.get("admin", {})
                authors = admin.get("authors_list", {}).get("author", [])

                for c in authors:

                    # ----- Case 1: dict author -----
                    if isinstance(c, dict):
                        inst = c.get("instance_type", "")
                        name = c.get("valueOf_")

                        # Only store first example of each type
                        if inst not in results:
                            results[inst] = name or "<no-valueOf_>"

                    # ----- Case 2: string author -----
                    elif isinstance(c, str):
                        inst = "<string-author>"
                        if inst not in results:
                            results[inst] = c.strip()

                    # ----- Case 3: unexpected -----
                    else:
                        inst = f"<unknown-{type(c).__name__}>"
                        if inst not in results:
                            results[inst] = str(c)

    return results

In [25]:
scan_author_instance_types_with_examples(emdb_record_folder)

{'author': 'Markert J', '<string-author>': 'Jones MJ'}